In [213]:
import numpy as np
import gdstk as tk

from IPython.display import SVG

from gemkid.ue1 import layers
from gemkid import geometry as geom
from gemkid import mecstyle as mec

def sc(cell, scale=1):
    cell.write_svg("/tmp/blop.svg", background="#ffffff", scaling = scale)
    display(SVG("/tmp/blop.svg"))


In [214]:
dummycell = tk.Cell("DummyCell")

In [215]:
array = tk.read_gds("../ue2-ucsb-array.gds")
tm = tk.read_gds("../ue2-ucsb-tm4.gds")

tms = [c for c in tm.cells if "tm4" in c.name]
arrays = [c for c in array.cells if "tm4" in c.name]
([t.name for t in tms], [a.name for a in arrays])

filtered = {}
for toplevel in arrays + tms:
    filtered[toplevel.name] = {}
    for c in toplevel.references:
        if "crossovers" in c.cell.name:
            filtered[toplevel.name][c.cell.name] = (c.cell, c.origin)
            c.cell = dummycell

tms = [t.flatten() for t in tms]
array = [a.flatten() for a in arrays]

In [216]:
filtered

{'tm4-array-ucsb-8ph-v0': {'crossovers-v0': (<gdstk.Cell at 0x10f1a7330>,
   (1332.0, 0.0))},
 'tm4-array-ucsb-8ph-v1': {'crossovers-v1': (<gdstk.Cell at 0x10d67ceb0>,
   (1332.0, 0.0))},
 'tm4-array-ucsb-8ph-v2': {'crossovers-v2': (<gdstk.Cell at 0x10ddca590>,
   (1332.0, 0.0))},
 'tm4-array-ucsb-8ph-v3': {'crossovers-v3': (<gdstk.Cell at 0x10db418d0>,
   (1332.0, 0.0))},
 'tm4-tm-ucsb-8ph-v0': {'half_coax_crossovers0': (<gdstk.Cell at 0x10dbd6cf0>,
   (0.0, 0.0)),
  'crossovers0': (<gdstk.Cell at 0x10dbd9630>, (0.0, 0.0))},
 'tm4-tm-ucsb-8ph-v1': {'half_coax_crossovers1': (<gdstk.Cell at 0x10d304690>,
   (0.0, 0.0)),
  'crossovers1': (<gdstk.Cell at 0x10d36edd0>, (0.0, 0.0))},
 'tm4-tm-ucsb-8ph-v2': {'half_coax_crossovers2': (<gdstk.Cell at 0x10d3a46b0>,
   (0.0, 0.0)),
  'crossovers2': (<gdstk.Cell at 0x10d362fd0>, (0.0, 0.0))},
 'tm4-tm-ucsb-8ph-v3': {'half_coax_crossovers3': (<gdstk.Cell at 0x10d3e94d0>,
   (0.0, 0.0)),
  'crossovers3': (<gdstk.Cell at 0x10d3b98d0>, (0.0, 0.0))}}

In [227]:
def filter_polys(cell, layer=(0, 0)):
    l = []
    for p in cell.get_polygons():
        if p.layer == layer[0] and p.datatype == layer[1]:
            l.append(p)
    return l

def translate_polys(ps, origin = (0, 0)):
    return [p.translate(origin) for p in ps]

def rotate_polys(ps, angle=np.pi/2, origin=(0, 0)):
    return [p.rotate(angle, origin) for p in ps]

def mask(polys, width, pos=0):
    rect = tk.rectangle((- width / 2 + pos, -3000), (width / 2 + pos, 3000))
    return tk.boolean(rect, polys, "and", 0.0001)

In [229]:
glassplate = tk.Cell("GlassPlate")
tm_shift = 1250-3000

polys = []

# AtA
polys.extend(
    translate_polys(mask(filter_polys(array[0]), 6000), (0, 6200))
)
polys.extend(
    translate_polys(mask(translate_polys(filter_polys(tms[0]), (0, tm_shift)), 6000), (6200, 6200))
)
# Hf
polys.extend(
    translate_polys(mask(filter_polys(array[0], (1, 0)), 1000), (-3000 + 500, 0))
)
polys.extend(
    translate_polys(mask(filter_polys(array[0], (1, 0)), 1000, -2500), (-2000 + 2500 + 500, 0))
)
polys.extend(
    translate_polys(mask(filter_polys(array[0], (1, 0)), 1000, +2500), (-2000 - 2500 + 500, 0))
)
polys.extend(
    translate_polys(mask(translate_polys(filter_polys(tms[0], (1, 0)), (0, tm_shift)), 1000), (6200 - 3000 + 500 + 1000, 0))
)
polys.extend(
    translate_polys(mask(translate_polys(filter_polys(tms[0], (1, 0)), (0, tm_shift)), 1000, -2000), (6200 - 3000 + 500 + 2000, 0))
)

# Contact
polys.extend(
    translate_polys(mask(filter_polys(array[0], (2, 0)), 1000), (-3000 + 500, 6200 + 6200))
)
polys.extend(
    translate_polys(mask(filter_polys(array[1], (2, 0)), 1000), (-2000 + 500, 6200 + 6200))
)
polys.extend(
    translate_polys(mask(filter_polys(array[2], (2, 0)), 1000), (-1000 + 500, 6200 + 6200))
)
polys.extend(
    translate_polys(mask(filter_polys(array[3], (2, 0)), 1000), (-0000 + 500, 6200 + 6200))
)

polys.extend(
    translate_polys(mask(translate_polys(filter_polys(tms[0], (2, 0)), (0, tm_shift)), 1000), (6200 - 3000 + 500, 6200 + 6200))
)
polys.extend(
    translate_polys(mask(translate_polys(filter_polys(tms[1], (2, 0)), (0, tm_shift)), 1000), (6200 - 2000 + 500, 6200 + 6200))
)
polys.extend(
    translate_polys(mask(translate_polys(filter_polys(tms[2], (2, 0)), (0, tm_shift)), 1000), (6200 - 1000 + 500, 6200 + 6200))
)
polys.extend(
    translate_polys(mask(translate_polys(filter_polys(tms[3], (2, 0)), (0, tm_shift)), 1000), (6200 - 0000 + 500, 6200 + 6200))
)

# Contact Liftoff
polys.extend(
    translate_polys(mask(filter_polys(array[0], (3, 0)), 1000), (1000 + 500, 6200 + 6200))
)
polys.extend(
    translate_polys(mask(filter_polys(array[2], (3, 0)), 1000), (2000 + 500, 6200 + 6200))
)
polys.extend(
    translate_polys(mask(translate_polys(filter_polys(tms[0], (3, 0)), (0, tm_shift)), 1000), (6200 + 1000 + 500, 6200 + 6200))
)
polys.extend(
    translate_polys(mask(translate_polys(filter_polys(tms[2], (3, 0)), (0, tm_shift)), 1000), (6200 + 2000 + 500, 6200 + 6200))
)

# ASI
polys.extend(
    translate_polys(mask(filter_polys(array[2], (4, 0)), 1000), (1000 + 500, 0))
)
polys.extend(
    translate_polys(mask(filter_polys(array[2], (4, 1)), 1000), (2000 + 500, 0))
)
polys.extend(
    translate_polys(mask(translate_polys(filter_polys(tms[2], (4, 0)), (0, tm_shift)), 1000), (6200 + 1000 + 500, 0))
)
polys.extend(
    translate_polys(mask(translate_polys(filter_polys(tms[2], (4, 1)), (0, tm_shift)), 1000), (6200 + 2000 + 500, 0))
)

# Half Coax
polys.extend(
    translate_polys(mask(translate_polys(filter_polys(filtered['tm4-tm-ucsb-8ph-v0']['half_coax_crossovers0'][0], (1, 0)), (0, tm_shift)), 1000), (3000 + 200 + 6000 + 500, 6200))
)
polys.extend(
    translate_polys(mask(translate_polys(filter_polys(filtered['tm4-tm-ucsb-8ph-v0']['half_coax_crossovers0'][0], (2, 0)), (0, tm_shift)), 1000), (3000 + 200 + 6000 + 500, 0))
)

# Crossovers
polys.extend(
    translate_polys(mask(translate_polys(filter_polys(filtered['tm4-tm-ucsb-8ph-v0']['crossovers0'][0], (1, 0)), (0, tm_shift)), 1000), (-1000 + 500, 0))
)
polys.extend(
    translate_polys(mask(translate_polys(filter_polys(filtered['tm4-tm-ucsb-8ph-v0']['crossovers0'][0], (2, 0)), (0, tm_shift)), 1000), (-0000 + 500, 0))
)

polys.extend(
    translate_polys(mask(translate_polys(filter_polys(filtered['tm4-array-ucsb-8ph-v0']['crossovers-v0'][0], (1, 0)), (0, 0)), 1000), (6200 - 1000 + 500, 0))
)
polys.extend(
    translate_polys(mask(translate_polys(filter_polys(filtered['tm4-array-ucsb-8ph-v0']['crossovers-v0'][0], (2, 0)), (0, 0)), 1000), (6200 - 0000 + 500, 0))
)

# Soldermask
polys.append(
    tk.ellipse(
        (3000 + 200 + 6000 + 500, 6200 + 3000 + 200),
        125.0,
        tolerance=1,
    )
)

letters = "0123456789ABCDEF#-/V"

for i in range(0, 4):
    for j in range(0, 5):
        polys.extend(tk.text(letters[i + j * 4], 72, (3000 + 200 + 6000 + 500 - 300 + 200 * i, 6200 + 3000 + 400 + 200 * j)))

polys = translate_polys(polys, (3000 - 6000 - 200 - 6000 - 1000, 3000 + 100))
polys_inverse = tk.boolean(
    tk.rectangle((0, 0), (13200, 6000 + 200 + 6000 + 200 + 6000 + 200)),
    translate_polys([p.copy() for p in polys], (13200 + 100, 0)),
    "not",
    0.0001
)
glassplate.add(*rotate_polys(translate_polys(polys, (0, -18600/2))))
glassplate.add(*rotate_polys(translate_polys(polys_inverse, (0, -18600/2))))
glassplate.add(
    *rotate_polys(tk.boolean(
        tk.rectangle((-27000 / 2, -22000 / 2), (27000 / 2, 22000 / 2)),
        tk.ellipse((0, 0), 31000 / 2),
        "and",
        1.0,
        99,
        99
    )),
)

lib = tk.Library()
lib.add(glassplate)
lib.write_gds("./ue2-ucsb-print-for-digidat.gds")
#sc(glassplate, 0.07)